# FreVA: Food Waste to Platform Chemicals (LP Model)

Linear programming model that allocates food-loss streams to platform chemicals. Built by Yujun Wei (WUR). NaaVRE-ready version: solver path no longer hardcoded, dataset and objective exposed as parameters, intermediate dicts pickled to `/tmp/data/`, solution values passed as JSON, results written to Cloud Storage as CSV and JSON.

Cell 1 holds the `param_` variables (top params cell, not containerised). Cells 2 through 4 are containerised.

Why building and solving live in the same cell: a Pyomo `ConcreteModel` cannot be pickled across container boundaries because `pickle.load` requires the constraint rule functions to be importable from `__main__`, and each container has its own `__main__`. Keeping build and solve in one container avoids the cross-container model handoff. Only solution values (floats and ints) leave the solve_lp cell.

In [1]:
# params
# Top params cell. Not containerised. Holds all runtime parameters.

# Region: "CN", "EU", or "US". Controls which FL2CH dataset is loaded when
# param_input_xlsx_path is empty.
param_region = "CN"

# Explicit path to the FL2CH Excel input. Default points at Cloud Storage,
# which is the only location both JupyterLab and the workflow containers
# can see. To run locally with a data/ folder next to the notebook, set this
# to e.g. "data/FL2CH_CN.xlsx".
param_input_xlsx_path = "/home/jovyan/Cloud Storage/naa-vre-public/vl-bgis-urban-planning/FL2CH_CN.xlsx"

# Objective: "profit" maximises revenue, "gwp" maximises GWP saving.
param_objective = "profit"

# Where to write outputs. Default points at Cloud Storage so workflow runs work.
param_output_csv_path = "/home/jovyan/Cloud Storage/naa-vre-user-data/freva_results.csv"
param_output_summary_path = "/home/jovyan/Cloud Storage/naa-vre-user-data/freva_summary.json"


In [2]:
# data_loader_FreVA
# Reads the five sheets from the FL2CH Excel file, converts them into the
# dict structures Pyomo wants, pickles them to /tmp/data/ for the next cell.

import pandas as pd
import pickle
import os

# Resolve input path. If param_input_xlsx_path is non-empty, use it directly
# (default: Cloud Storage path). Otherwise build data/FL2CH_<region>.xlsx for
# local notebook execution. Strip stray quotes (NaaVRE wraps strings in extra quotes).
region = str(param_region).strip('"').strip("'").upper()
explicit_path = str(param_input_xlsx_path).strip('"').strip("'")
objective_choice = str(param_objective).strip('"').strip("'").lower()

if explicit_path:
    xlsx_path = explicit_path
else:
    if region not in ("CN", "EU", "US"):
        raise ValueError(f"param_region must be CN, EU, or US, got {region!r}")
    xlsx_path = os.path.join("data", f"FL2CH_{region}.xlsx")

if objective_choice not in ("profit", "gwp"):
    raise ValueError(f"param_objective must be 'profit' or 'gwp', got {objective_choice!r}")

# eta: yield matrix (I rows, J columns). FL: food-loss quantities (I).
# CH: chemical demand (J). profit / GWP: per-chemical objective coefficients (J).
eta_df = pd.read_excel(xlsx_path, sheet_name="eta", index_col=0)
FL_df = pd.read_excel(xlsx_path, sheet_name="FL")
CH_df = pd.read_excel(xlsx_path, sheet_name="CH")
profit_df = pd.read_excel(xlsx_path, sheet_name="profit")
GWP_df = pd.read_excel(xlsx_path, sheet_name="GWP")

# Pyomo wants 1-indexed dicts. eta has tuple keys (i, j); the others have int keys.
eta_dict = {(r, c): v for r, row in eta_df.to_dict(orient="index").items() for c, v in row.items()}
FL_dict = {i + 1: FL_df["Column1"].to_dict()[i] for i in FL_df["Column1"].to_dict()}
CH_dict = {i + 1: CH_df["Column1"].to_dict()[i] for i in CH_df["Column1"].to_dict()}
profit_dict = {i + 1: profit_df["Column1"].to_dict()[i] for i in profit_df["Column1"].to_dict()}
GWP_dict = {i + 1: GWP_df["Column1"].to_dict()[i] for i in GWP_df["Column1"].to_dict()}

# Counts always come from the sheets.
n_I = len(FL_dict)
n_J = len(CH_dict)

# Pickle everything to /tmp/data/. Downstream cell receives file paths as strings.
os.makedirs("/tmp/data", exist_ok=True)

eta_path = "/tmp/data/eta_dict.pkl"
fl_path = "/tmp/data/FL_dict.pkl"
ch_path = "/tmp/data/CH_dict.pkl"
profit_path = "/tmp/data/profit_dict.pkl"
gwp_path = "/tmp/data/GWP_dict.pkl"

with open(eta_path, "wb") as f:
    pickle.dump(eta_dict, f)
with open(fl_path, "wb") as f:
    pickle.dump(FL_dict, f)
with open(ch_path, "wb") as f:
    pickle.dump(CH_dict, f)
with open(profit_path, "wb") as f:
    pickle.dump(profit_dict, f)
with open(gwp_path, "wb") as f:
    pickle.dump(GWP_dict, f)

print(f"Loaded {xlsx_path}")
print(f"Region: {region}")
print(f"Objective: {objective_choice}")
print(f"I = {n_I} food-loss streams, J = {n_J} platform chemicals")
print(f"Pickled to /tmp/data/: eta, FL, CH, profit, GWP")

Loaded /home/jovyan/Cloud Storage/naa-vre-public/vl-bgis-urban-planning/FL2CH_CN.xlsx
Region: CN
Objective: profit
I = 22 food-loss streams, J = 11 platform chemicals
Pickled to /tmp/data/: eta, FL, CH, profit, GWP


In [3]:
# solve_lp
# Reads the pickled dicts, builds the Pyomo LP, solves with GLPK, writes
# solution values to /tmp/data/ as JSON. Building and solving happen in the
# same container because a pickled ConcreteModel cannot be unpickled in a
# different container (the constraint rule functions live in the building
# container's __main__ and are not importable elsewhere).

import pickle
import json
from pyomo.environ import ConcreteModel, RangeSet, Param, Var, Objective, Constraint, NonNegativeReals, maximize, SolverFactory

with open(eta_path, "rb") as f:
    eta_dict = pickle.load(f)
with open(fl_path, "rb") as f:
    FL_dict = pickle.load(f)
with open(ch_path, "rb") as f:
    CH_dict = pickle.load(f)
with open(profit_path, "rb") as f:
    profit_dict = pickle.load(f)
with open(gwp_path, "rb") as f:
    GWP_dict = pickle.load(f)

model = ConcreteModel()

model.I = RangeSet(1, n_I)
model.J = RangeSet(1, n_J)

model.FL = Param(model.I, initialize=FL_dict)
model.CH = Param(model.J, initialize=CH_dict)
model.eta = Param(model.I, model.J, initialize=eta_dict)
model.profit = Param(model.J, initialize=profit_dict)
model.GWP = Param(model.J, initialize=GWP_dict)

model.x = Var(model.I, within=NonNegativeReals)               # total FL[i] used
model.y = Var(model.J, within=NonNegativeReals)               # CH[j] produced
model.w = Var(model.I, model.J, within=NonNegativeReals)      # FL[i] used for CH[j]

def profit_obj(m):
    return sum(m.y[j] * m.profit[j] for j in m.J)

def gwp_obj(m):
    return sum(m.y[j] * m.GWP[j] for j in m.J)

if objective_choice == "profit":
    model.obj = Objective(rule=profit_obj, sense=maximize)
else:
    model.obj = Objective(rule=gwp_obj, sense=maximize)

def x_constraint_rule(m, i):
    return m.x[i] == sum(m.w[i, j] for j in m.J)

def y_constraint_rule(m, j):
    return m.y[j] == sum(m.w[i, j] * m.eta[i, j] for i in m.I)

def y_demand_rule(m, j):
    return m.y[j] <= m.CH[j]

def x_supply_rule(m, i):
    return m.x[i] <= m.FL[i]

model.x_constraint = Constraint(model.I, rule=x_constraint_rule)
model.y_constraint = Constraint(model.J, rule=y_constraint_rule)
model.y_demand = Constraint(model.J, rule=y_demand_rule)
model.x_supply = Constraint(model.I, rule=x_supply_rule)

# Solve. SolverFactory finds glpsol on PATH. Requires `glpk` in the flavour's environment.yaml.
solver = SolverFactory("glpk")
if not solver.available():
    raise RuntimeError("GLPK solver not found. Add `glpk` to the bgis-urban-planning flavour's environment.yaml.")

results = solver.solve(model, tee=False)

solver_status = str(results.solver.status)
term_condition = str(results.solver.termination_condition)
obj_value = float(model.obj())

# Extract solution values as plain Python types. Save to JSON for the results writer.
solution = {
    "x": {int(i): float(model.x[i].value or 0.0) for i in model.I},
    "y": {int(j): float(model.y[j].value or 0.0) for j in model.J},
    "w": [
        {"i": int(i), "j": int(j), "w": float(model.w[i, j].value or 0.0), "eta": float(model.eta[i, j])}
        for i in model.I for j in model.J
    ],
}

solution_path = "/tmp/data/freva_solution.json"
with open(solution_path, "w") as f:
    json.dump(solution, f)

print(f"Solver status: {solver_status}")
print(f"Termination: {term_condition}")
print(f"Objective value ({objective_choice}): {obj_value:,.2f}")
print(f"Wrote solution to {solution_path}")

Solver status: ok
Termination: optimal
Objective value (profit): 27,092,387,898.71
Wrote solution to /tmp/data/freva_solution.json


In [4]:
# results_writer
# Reads the solution JSON, writes a CSV of the allocation and a JSON summary
# to Cloud Storage. Replaces the original model.display() dump with something
# downstream tools can consume.

import json
import pandas as pd

# Strip stray quotes from every string crossing a cell boundary. NaaVRE wraps
# strings in extra quotes when passing them between containerised cells.
solution_path = str(solution_path).strip().strip('"').strip("'").strip()
region = str(region).strip().strip('"').strip("'").strip()
objective_choice = str(objective_choice).strip().strip('"').strip("'").strip()
solver_status = str(solver_status).strip().strip('"').strip("'").strip()
term_condition = str(term_condition).strip().strip('"').strip("'").strip()
csv_path = str(param_output_csv_path).strip().strip('"').strip("'").strip()
summary_path = str(param_output_summary_path).strip().strip('"').strip("'").strip()

with open(solution_path, "r") as f:
    solution = json.load(f)

rows = []
for entry in solution["w"]:
    flow = entry["w"]
    if flow > 1e-9:
        rows.append({
            "fl_stream": entry["i"],
            "chemical": entry["j"],
            "flow": flow,
            "yield_eta": entry["eta"],
            "chemical_produced": flow * entry["eta"],
        })
alloc_df = pd.DataFrame(rows)
alloc_df.to_csv(csv_path, index=False)

summary = {
    "region": region,
    "objective": objective_choice,
    "objective_value": obj_value,
    "solver_status": solver_status,
    "termination_condition": term_condition,
    "n_fl_streams": int(n_I),
    "n_chemicals": int(n_J),
    "fl_used": {int(k): v for k, v in solution["x"].items()},
    "ch_produced": {int(k): v for k, v in solution["y"].items()},
}
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Wrote allocation CSV: {csv_path} ({len(alloc_df)} rows)")
print(f"Wrote summary JSON:  {summary_path}")
print(f"\nFL usage (top 5 by quantity):")
print(alloc_df.groupby("fl_stream")["flow"].sum().sort_values(ascending=False).head())

Wrote allocation CSV: /home/jovyan/Cloud Storage/naa-vre-user-data/freva_results.csv (14 rows)
Wrote summary JSON:  /home/jovyan/Cloud Storage/naa-vre-user-data/freva_summary.json

FL usage (top 5 by quantity):
fl_stream
1     10144.234150
8      8127.254509
5      7749.544236
17     3808.227921
15     2566.197863
Name: flow, dtype: float64
